# TripoSR with OpenLRM Training Pipeline

This notebook demonstrates how to train TripoSR models using the OpenLRM training pipeline by cloning the specific `openlrm_training` branch.

## Setup Environment

First, let's install the necessary dependencies.

In [ ]:
!pip install torch torchvision safetensors omegaconf accelerate tensorboard

## Clone Repository

Next, let's clone the OpenLRM repository and your specific TripoSR branch.

In [ ]:
# Clone OpenLRM repository
!git clone https://github.com/3DTopia/OpenLRM.git
%cd OpenLRM
!pip install -e .
%cd ..


# Clone your TripoSR repository with the specific branch
!git clone -b openlrm_training --single-branch --depth 1 https://github.com/wirapratamaz/TripoSR.git

In [ ]:
# Verify OpenLRM installation
import sys
try:
    import openlrm
    print("✅ OpenLRM is installed successfully!")
    print(f"OpenLRM version: {openlrm.__version__ if hasattr(openlrm, '__version__') else 'development'}")
except ImportError:
    print("❌ OpenLRM installation failed. Let's try a different approach.")
    # Alternative installation method
    !pip install git+https://github.com/3DTopia/OpenLRM.git
    # Verify again
    try:
        import openlrm
        print("✅ OpenLRM is now installed!")
    except ImportError:
        print("❌ OpenLRM installation still failed. Please check the repository structure.")

## Prepare Dataset

Upload your dataset or use a provided one. Here we'll check if a dataset exists or create a simple one for demonstration.

In [ ]:
import os
import torch
import numpy as np
from PIL import Image

# Check if dataset already exists
dataset_path = "TripoSR/dataset"
if os.path.exists(dataset_path) and len(os.listdir(dataset_path)) > 0:
    print(f"Dataset found at {dataset_path}")
else:
    print("Creating a demo dataset...")
    # Create dataset directories
    !mkdir -p TripoSR/dataset/train/sample1 TripoSR/dataset/val/sample1
    
    # Create synthetic samples for demonstration
    for split in ['train', 'val']:
        for i in range(1, 4):  # Create 3 samples
            sample_dir = f"TripoSR/dataset/{split}/sample{i}"
            os.makedirs(sample_dir, exist_ok=True)
            
            # Create a random RGB image
            img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(img)
            img.save(f"{sample_dir}/rgb.png")
            
            # Create a simple mask
            mask = np.zeros((224, 224), dtype=np.uint8)
            mask[50:150, 50:150] = 255  # White square in the middle
            mask = Image.fromarray(mask)
            mask.save(f"{sample_dir}/mask.png")
    
    print("Demo dataset created!")

## Upload Your Dataset (Optional)

If you have a specific dataset you want to use, use this cell to upload it. This is especially useful if your instructor provided a specific dataset.

In [ ]:
from google.colab import files

# Uncomment the following lines if you want to upload your own dataset
# print("Please upload your dataset files (you can select multiple files)...")
# uploaded = files.upload()
# 
# # Process uploaded files
# import shutil
# for filename in uploaded.keys():
#     # Define the target path based on your dataset structure
#     target_path = f"TripoSR/dataset/{filename}"
#     shutil.move(filename, target_path)
#     print(f"Moved {filename} to {target_path}")

## Configure Training

Let's modify the configuration for our Colab training run.

In [ ]:
from omegaconf import OmegaConf

# Check if the config file exists
config_path = 'TripoSR/openlrm_integration/configs/default_config.yaml'
if os.path.exists(config_path):
    # Load the default configuration
    cfg = OmegaConf.load(config_path)
    
    # Modify for Colab environment
    cfg.data.dataset_path = './TripoSR/dataset'
    cfg.train.batch_size = 2  # Smaller batch size for Colab
    cfg.train.epochs = 10     # Fewer epochs for Colab
    cfg.val.eval_global_steps = 10  # Evaluate more frequently
    cfg.saver.checkpoint_global_steps = 20  # Save checkpoints more frequently
    cfg.logger.trackers = ["tensorboard"]  # Only use tensorboard for logging
    
    # Save the modified configuration
    OmegaConf.save(cfg, 'TripoSR/openlrm_integration/configs/colab_config.yaml')
    
    print("Configuration updated for Colab environment!")
    print(OmegaConf.to_yaml(cfg))
else:
    print(f"Config file not found at {config_path}. Please check the repository structure.")

## Install Additional Requirements

Make sure all required packages are installed.

In [ ]:
!pip install -r TripoSR/requirements.txt

# Install additional dependencies that OpenLRM might need
!pip install accelerate safetensors einops diffusers transformers


In [ ]:
# Update PYTHONPATH to include OpenLRM
import sys
if '/content/OpenLRM' not in sys.path:
    sys.path.append('/content/OpenLRM')
sys.path.append('/content/TripoSR')

# Check if openlrm is accessible
try:
    from openlrm.utils.logging import configure_logger, get_logger
    print("✅ OpenLRM modules are accessible!")
except ImportError as e:
    print(f"❌ Error importing OpenLRM modules: {e}")

## Start Training

Now, let's start the training process.

In [ ]:
import sys
sys.path.append('./TripoSR')

# Change to the TripoSR openlrm_integration directory
%cd TripoSR/openlrm_integration

# Run the training script with the export_ckpt flag to generate .ckpt files
!python train.py --config configs/colab_config.yaml --export_ckpt

## Monitor Training with TensorBoard

Let's use TensorBoard to monitor the training progress.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./runs

## Check Exported Model

After training, the model should be exported to a .ckpt file. Let's verify that it exists.

In [ ]:
# Find the .ckpt files
import glob

ckpt_files = glob.glob('./models/triposr/openlrm-mix-base/*.ckpt')
if ckpt_files:
    print("Found .ckpt files:")
    for f in ckpt_files:
        print(f"- {f}")
    
    # Get size of the checkpoint file
    import os
    size_mb = os.path.getsize(ckpt_files[0]) / (1024 * 1024)
    print(f"Checkpoint size: {size_mb:.2f} MB")
else:
    print("No .ckpt files found. Training might not have completed successfully.")

## Download the Trained Model

Let's download the trained model .ckpt file so you can use it with TripoSR reconstruction.

In [ ]:
from google.colab import files

ckpt_files = glob.glob('./models/triposr/openlrm-mix-base/*.ckpt')
if ckpt_files:
    # Download the latest checkpoint file
    latest_ckpt = max(ckpt_files, key=os.path.getctime)
    print(f"Downloading {latest_ckpt}...")
    files.download(latest_ckpt)
else:
    print("No .ckpt files found to download.")

## Next Steps

Now that you have trained a model and exported it as a .ckpt file, you can use this file with the TripoSR reconstruction pipeline. Here's what you should do:

1. Download the .ckpt file from this Colab notebook
2. Place the .ckpt file in your local TripoSR project directory
3. Update your TripoSR configuration to use this model for reconstruction
4. Run the reconstruction process with your new model

Remember, as per your instructor's guidance, you should:
- Use OpenLRM for training to generate model .ckpt files (which you just did)
- Continue using TripoSR for the reconstruction process
- Replace the model.ckpt in TripoSR with the one generated through OpenLRM training